# MODIS + VIIRS Hotspot Harmonization — Canada (2002–2025)
**NASA Space Apps — Challenge 8: Harmonization of MODIS and VIIRS Hot Spots**

`Harmonized Table.csv` is the stacked FIRMS archive: **MODIS Aqua (2002–2025, 1 km)** and **VIIRS S-NPP (2012–2025, 375 m)**.
Stacking them is not enough on its own: VIIRS pixels are ~7× smaller in area and it sees far more small fires, so raw counts jump up when VIIRS arrives in 2012.

This notebook builds one consistent record and a burning activity calendar from it:

1. **Load & clean**: compact dtypes, one confidence scale, vegetation fires only
2. **Common grid**: both sensors binned to the same 0.1° × daily "fire cells", which cancels most of the pixel size difference
3. **Overlap calibration (2012–2025)**: learn how many VIIRS fire cells correspond to one MODIS fire cell, by month of year
4. **Harmonized series 2002–2025**: VIIRS where it exists, calibrated MODIS before it
5. **Burning activity calendar** for any area of interest (AOI)
6. **Climatology, anomalies and critical periods**
7. **Red-pixel fire map**
8. **Export** for the web app

> Every setting lives in the **Config** cell. Change the AOI or the years there and re-run everything below it.

## 0. Setup & config

In [ ]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# ============================ CONFIG ============================
# The notebook sits next to the CSV. Override with the FIRE_CSV env var if needed.
DATA_PATH = Path(os.environ.get("FIRE_CSV", "HarmonizedTable.csv"))
OUT_DIR   = Path("outputs")            # exports go here (created automatically)

GRID_DEG  = 0.1                        # common grid size in degrees (~11 km N-S)
TARGET    = "VIIRS"                    # harmonize onto the "VIIRS" scale or the "MODIS" scale
MIN_CONF  = "n"                        # keep "n"(nominal)+"h"(high); use "l" to keep everything
FIRE_TYPES = [0]                       # FIRMS type 0 = presumed vegetation fire (drops volcanoes, static land, offshore)

# Area of interest: pick a preset name, or set AOI = (lat_min, lat_max, lon_min, lon_max)
AOI_PRESETS = {                        # rough bounding boxes, not exact borders
    "Canada":           (41.5, 84.0, -141.5, -52.0),
    "British Columbia": (48.2, 60.0, -139.1, -114.0),
    "Alberta":          (49.0, 60.0, -120.0, -110.0),
    "Saskatchewan":     (49.0, 60.0, -110.0, -101.4),
    "Manitoba":         (49.0, 60.0, -102.0,  -88.9),
    "Ontario":          (41.7, 56.9,  -95.2,  -74.3),
    "Quebec":           (45.0, 62.6,  -79.8,  -57.1),
    "Northwest Territories": (60.0, 78.8, -136.5, -102.0),
}
AOI_NAME = "Canada"
AOI = AOI_PRESETS[AOI_NAME]

MAP_YEAR = 2023                        # year shown on the red-pixel map
# ================================================================
OUT_DIR.mkdir(exist_ok=True)
print("Data file:", DATA_PATH.resolve(), "| exists:", DATA_PATH.exists())

## 1. Load
The file is about 870 MB and 6.7 M rows, so it is read in chunks with only the needed columns and compact dtypes. That keeps the table under about 300 MB in RAM.
`acq_date` is parsed from its leading `YYYY-MM-DD`, which is much faster than generic date parsing.

In [ ]:
USECOLS = ["latitude", "longitude", "brightness", "acq_date", "acq_time", "instrument",
           "confidence", "frp", "daynight", "type"]
DTYPES  = {"latitude": "float32", "longitude": "float32", "brightness": "float32",
           "acq_date": "string", "acq_time": "int16", "instrument": "category",
           "confidence": "string", "frp": "float32", "daynight": "category", "type": "int8"}

def conf_class(s: pd.Series) -> pd.Series:
    num = pd.to_numeric(s, errors="coerce").astype("float64")
    out = s.astype(object).str.lower().str[0]                     # VIIRS letters
    out = out.where(num.isna(),
                    pd.Series(np.select([num.values < 30, num.values < 80], ["l", "n"], "h"), index=s.index))  # MODIS %
    return pd.Categorical(out, categories=["l", "n", "h"])

# MODIS confidence (0-100 %) -> l/n/h, using FIRMS thresholds (details in section 2)
t0 = time.time()
parts = []
for chunk in pd.read_csv(DATA_PATH, usecols=USECOLS, dtype=DTYPES, chunksize=1_000_000):
    chunk["acq_date"] = pd.to_datetime(chunk["acq_date"].str[:10], format="%Y-%m-%d")
    chunk["conf_cls"] = conf_class(chunk.pop("confidence"))
    parts.append(chunk)
raw = pd.concat(parts, ignore_index=True)
del parts
raw["sensor"] = raw.pop("instrument").astype(str).str.upper().astype("category")

print(f"{len(raw):,} rows loaded in {time.time()-t0:.0f}s  |  {raw.memory_usage(deep=True).sum()/1e6:,.0f} MB in memory")
raw.head()

In [ ]:
# What is in the stack?
raw["year"] = raw["acq_date"].dt.year.astype("int16")
overview = raw.groupby(["year", "sensor"], observed=True).size().unstack(fill_value=0)
display(overview.T)
print("Date range:", raw.acq_date.min().date(), "→", raw.acq_date.max().date())

## 2. Clean and use one confidence scale
The two sensors record confidence differently:

| Sensor | Field | Classes |
|---|---|---|
| MODIS (C6.1) | 0–100 % | low < 30, nominal 30–79, high ≥ 80 (FIRMS user guide) |
| VIIRS | letters | `l` low, `n` nominal, `h` high |

MODIS is mapped onto the VIIRS letters so one filter applies to both. Only vegetation fires (`type == 0`) are kept.

In [ ]:
# conf_cls was computed per chunk while loading (see conf_class in section 1)
print(pd.crosstab(raw["sensor"], raw["conf_cls"]), "\n")
print(pd.crosstab(raw["sensor"], raw["type"]), "\n")

keep_conf = {"l": ["l", "n", "h"], "n": ["n", "h"], "h": ["h"]}[MIN_CONF]
fires = raw[raw["conf_cls"].isin(keep_conf) & raw["type"].isin(FIRE_TYPES)].copy()
print(f"Kept {len(fires):,} of {len(raw):,} detections ({len(fires)/len(raw):.1%})")

## 3. Put both sensors on one grid (0.1° × day)
Raw detection counts can't be compared, because one 1 km MODIS pixel covers about seven 375 m VIIRS pixels.
Each detection is snapped to a 0.1° cell, and the unit becomes the **fire cell-day**: a cell with at least one detection that day.
Both sensors resolve a fire at that scale, so the 2012 step change largely disappears.
FRP (fire radiative power, MW) is summed per cell-day as an intensity measure.

In [ ]:
fires["cy"] = np.floor(fires["latitude"]  / GRID_DEG).astype("int32")
fires["cx"] = np.floor(fires["longitude"] / GRID_DEG).astype("int32")

cells = (fires.groupby(["sensor", "acq_date", "cy", "cx"], observed=True)
              .agg(n_det=("frp", "size"), frp_sum=("frp", "sum"))
              .reset_index())
cells["lat"] = (cells["cy"] + 0.5) * GRID_DEG
cells["lon"] = (cells["cx"] + 0.5) * GRID_DEG
print(f"{len(cells):,} fire cell-days")
print(cells.groupby("sensor", observed=True)
           .agg(cell_days=("n_det", "size"), det_per_cell_day=("n_det", "mean")).round(2))

## 4. Calibrate MODIS against VIIRS over the overlap (2012–2025)
Both sensors operated together for 14 years, and both fly at about 13:30 / 01:30 local time (Aqua and S-NPP), which makes them a well-matched pair.

**Method.** Count fire cells per day for each sensor. For each calendar month, the calibration factor is
$k_m = \sum \text{VIIRS cells} \;/\; \sum \text{MODIS cells}$ over every overlap day in that month.
This is a ratio estimator. It is robust to the many days where one sensor sees zero, and it allows for the seasonal shift between spring grass and agricultural burns (small, VIIRS-only) and summer crown fires (large, seen by both).
The cell-level agreement check and a hold-out test show how far to trust it.

In [ ]:
daily = (cells.groupby(["acq_date", "sensor"], observed=True).size()
              .unstack(fill_value=0)
              .reindex(columns=["MODIS", "VIIRS"], fill_value=0))
daily = daily.asfreq("D", fill_value=0)
viirs_start = cells.loc[cells.sensor == "VIIRS", "acq_date"].min()
overlap = daily.loc[viirs_start:].copy()
overlap["month"] = overlap.index.month
print("Overlap period:", viirs_start.date(), "→", overlap.index.max().date(), f"({len(overlap):,} days)")

SHRINK_N = 200   # MODIS cells at which a month's own k gets half the weight (also used in the hold-out test, 4c)

k_month = overlap.groupby("month")[["VIIRS", "MODIS"]].sum()
k_month["k"] = k_month["VIIRS"] / k_month["MODIS"].replace(0, np.nan)
k_global = overlap["VIIRS"].sum() / overlap["MODIS"].sum()
k_month["k"] = k_month["k"].fillna(k_global)
# months with very few MODIS cells give noisy ratios: shrink them toward the global factor
w = k_month["MODIS"] / (k_month["MODIS"] + SHRINK_N)
k_month["k_shrunk"] = w * k_month["k"] + (1 - w) * k_global
print(f"Global factor: 1 MODIS fire cell ≈ {k_global:.2f} VIIRS fire cells")
display(k_month.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
m = overlap[(overlap.MODIS > 0) | (overlap.VIIRS > 0)]
ax.scatter(m.MODIS + 1, m.VIIRS + 1, s=4, alpha=0.3, color="#c0392b")
lim = [1, max(m.MODIS.max(), m.VIIRS.max()) * 1.5]
ax.plot(lim, lim, "k--", lw=0.8, label="1 : 1")
ax.plot(lim, np.array(lim) * k_global, color="#2c3e50", lw=1.2, label=f"k = {k_global:.2f}")
ax.set(xscale="log", yscale="log", xlabel="MODIS fire cells / day (+1)", ylabel="VIIRS fire cells / day (+1)",
       title="Daily fire cells during the overlap")
ax.legend()

ax = axes[1]
ax.bar(k_month.index, k_month["k_shrunk"], color="#e67e22")
ax.axhline(k_global, color="#2c3e50", lw=1, ls="--")
ax.set(xticks=range(1, 13), xlabel="Month", ylabel="VIIRS / MODIS",
       title="Monthly calibration factor (shrunk)")
plt.tight_layout(); plt.show()

r = np.corrcoef(np.log1p(overlap.MODIS), np.log1p(overlap.VIIRS))[0, 1]
print(f"Correlation of log daily counts: r = {r:.3f}")

### 4b. Cell-level agreement
How often do the two sensors flag the **same** cell on the **same** day? VIIRS should catch nearly everything MODIS catches. The reverse is lower, because VIIRS sees many small fires that are below MODIS's detection limit.

In [ ]:
ov = cells[cells.acq_date >= viirs_start]
keys = {s: set(zip(g.acq_date.values, g.cy.values, g.cx.values)) for s, g in ov.groupby("sensor", observed=True)}
both = len(keys["MODIS"] & keys["VIIRS"])
agree = pd.Series({
    "MODIS cell-days":                len(keys["MODIS"]),
    "VIIRS cell-days":                len(keys["VIIRS"]),
    "Seen by both":                   both,
    "% of MODIS also seen by VIIRS":  100 * both / max(len(keys["MODIS"]), 1),
    "% of VIIRS also seen by MODIS":  100 * both / max(len(keys["VIIRS"]), 1),
    "Jaccard (both / either)":        both / max(len(keys["MODIS"] | keys["VIIRS"]), 1),
})
display(agree.round(3).to_frame("value"))
del keys

### 4c. Hold-out validation
Fit the factors on 2012–2018 only, apply them to MODIS for 2019 onward, and compare with what VIIRS actually saw. This is the error to expect when the factors are applied to the MODIS-only years (2002–2011).

In [ ]:
split = pd.Timestamp("2019-01-01")
train, test = overlap.loc[:split - pd.Timedelta(days=1)], overlap.loc[split:]
kt = train.groupby("month")[["VIIRS", "MODIS"]].sum()
kg = train.VIIRS.sum() / train.MODIS.sum()
kt_raw = (kt.VIIRS / kt.MODIS.replace(0, np.nan)).fillna(kg)
# same shrinkage as section 4, so the test checks the factors that section 5 actually applies
wt = kt.MODIS / (kt.MODIS + SHRINK_N)
kt = wt * kt_raw + (1 - wt) * kg
pred = test.MODIS * test.month.map(kt)

yr = pd.DataFrame({"VIIRS observed": test.VIIRS, "MODIS raw": test.MODIS, "MODIS calibrated": pred}).resample("YE").sum()
yr.index = yr.index.year
yr["error %"] = 100 * (yr["MODIS calibrated"] - yr["VIIRS observed"]) / yr["VIIRS observed"]
display(yr.round(0))

wk = pd.DataFrame({"obs": test.VIIRS, "pred": pred}).resample("W").sum()
print(f"Weekly correlation (hold-out): r = {np.corrcoef(wk.obs, wk.pred)[0,1]:.3f}")
print(f"Annual totals: median absolute error = {yr['error %'].abs().median():.1f}%")

## 5. Build the harmonized record (2002–2025)
For each cell-day there is a **harmonized weight** on the `TARGET` scale:

* `TARGET = "VIIRS"`: VIIRS cells count 1 from 2012 on. Before VIIRS existed, each MODIS cell counts that month's shrunk factor `k_shrunk` (section 4). MODIS cells from 2012 on are left out, so no fire is counted twice.
* `TARGET = "MODIS"`: MODIS cells count 1 for the whole record. VIIRS is not used here (only in the calibration and validation of section 4), so the series is purely MODIS-native.

`source` records where each row came from, so the web app can show it.

In [ ]:
kmap = k_month["k_shrunk"]
if TARGET == "VIIRS":
    pre  = cells[(cells.sensor == "MODIS") & (cells.acq_date < viirs_start)].copy()
    pre["weight"] = pre.acq_date.dt.month.map(kmap).astype("float32")
    pre["source"] = "MODIS (calibrated)"
    post = cells[(cells.sensor == "VIIRS")].copy()
    post["weight"] = np.float32(1.0)
    post["source"] = "VIIRS"
    harm = pd.concat([pre, post], ignore_index=True)
else:
    harm = cells[cells.sensor == "MODIS"].copy()
    harm["weight"] = np.float32(1.0)
    harm["source"] = "MODIS"

harm = harm.drop(columns="sensor")
harm["year"] = harm.acq_date.dt.year.astype("int16")
harm["doy"]  = harm.acq_date.dt.dayofyear.astype("int16")
harm["source"] = harm["source"].astype("category")
print(f"Harmonized cell-days: {len(harm):,}  (scale = {TARGET})")
harm.groupby("source", observed=True).weight.agg(["size", "sum"])

In [ ]:
# Before / after: annual totals for all of Canada
ann_raw = cells.groupby([cells.acq_date.dt.year, "sensor"], observed=True).size().unstack(fill_value=0)
ann_h   = harm.groupby("year").weight.sum()

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(ann_raw.index, ann_raw.get("MODIS"), "o-", color="#7f8c8d", label="MODIS raw cells")
ax.plot(ann_raw.index, ann_raw.get("VIIRS"), "s-", color="#2980b9", label="VIIRS raw cells")
ax.plot(ann_h.index, ann_h.values, "-", lw=3, color="#c0392b", alpha=0.85, label=f"Harmonized ({TARGET} scale)")
ax.axvline(viirs_start.year - 0.5, color="k", ls=":", lw=1)
ax.text(viirs_start.year - 0.4, ax.get_ylim()[1] * 0.92, "VIIRS starts", fontsize=9)
ax.set(title="Annual fire cell-days: raw sensors vs harmonized record", xlabel="Year", ylabel="Fire cell-days")
ax.legend(); plt.tight_layout(); plt.show()

## 6. Burning activity calendar for the AOI
Change `AOI_NAME` (or `AOI`) in the Config cell. The calendar shows the harmonized fire cells in every **week of every year**, so it is easy to see when the fire season starts, how long it lasts and which years stand out.

In [ ]:
def in_aoi(df, aoi=AOI):
    la0, la1, lo0, lo1 = aoi
    return df[(df.lat >= la0) & (df.lat <= la1) & (df.lon >= lo0) & (df.lon <= lo1)]

aoi_cells = in_aoi(harm)
aoi_daily = (aoi_cells.groupby("acq_date").weight.sum()
             .reindex(pd.date_range(f"{harm.year.min()}-01-01", f"{harm.year.max()}-12-31"), fill_value=0))
aoi_daily.index.name = "date"

cal = aoi_daily.to_frame("w")
cal["year"] = cal.index.year
cal["week"] = np.minimum((cal.index.dayofyear - 1) // 7 + 1, 52)
cal.loc[cal.index < harm.acq_date.min(), "w"] = np.nan        # before the record starts (Jul 2002) = no data, not zero fire
weekly = cal.groupby(["year", "week"]).w.sum(min_count=1).unstack()
print(f"AOI = {AOI_NAME} {AOI}: {len(aoi_cells):,} cell-days, {np.nansum(weekly.values):,.0f} harmonized fire cells")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
cmap_cal = plt.get_cmap("inferno").copy(); cmap_cal.set_bad("#dddddd")   # grey = no data
im = ax.imshow(weekly.values + 1, aspect="auto", cmap=cmap_cal, norm=mcolors.LogNorm(vmin=1),
               extent=[0.5, 52.5, weekly.index.max() + 0.5, weekly.index.min() - 0.5])
month_starts = pd.date_range("2021-01-01", periods=12, freq="MS")
ax.set_xticks((month_starts.dayofyear - 1) / 7 + 1, month_starts.strftime("%b"))
ax.set_yticks(weekly.index)
ax.axhline(viirs_start.year - 0.5, color="cyan", lw=1, ls="--")
ax.set(title=f"Burning activity calendar: {AOI_NAME} (harmonized fire cells per week, log scale)",
       xlabel="Week of year", ylabel="Year")
cb = plt.colorbar(im, ax=ax, pad=0.01); cb.set_label("fire cells / week (+1)")
plt.tight_layout(); plt.show()

## 7. Climatology, anomalies and critical periods
* **Climatology**: median and 10–90 % range of weekly activity across all years
* **Anomaly**: each week of each year compared with that week's climatology, as a robust z-score on log counts using median and MAD, so 2023 doesn't distort the baseline
* **Critical period**: weeks whose median activity is in the top quartile of the season, i.e. the window when monitoring matters most
* **Season timing**: for each year, the dates when 10 %, 50 % and 90 % of that year's activity had happened

In [ ]:
logw = np.log1p(weekly)
med  = logw.median()
mad  = (logw - med).abs().median().replace(0, np.nan) * 1.4826
z    = ((logw - med) / mad).fillna(0).clip(-6, 6).where(weekly.notna())

clim = pd.DataFrame({"median": weekly.median(), "p10": weekly.quantile(0.1), "p90": weekly.quantile(0.9)})
crit_thresh = clim["median"][clim["median"] > 0].quantile(0.75)
critical_weeks = clim.index[clim["median"] >= crit_thresh]

def week_to_date(w):   # label a week by its first day (non-leap year)
    return (pd.Timestamp("2021-01-01") + pd.Timedelta(days=7 * (int(w) - 1))).strftime("%d %b")

print(f"Critical period for {AOI_NAME}: weeks {critical_weeks.min()}–{critical_weeks.max()} "
      f"(≈ {week_to_date(critical_weeks.min())} to {week_to_date(critical_weeks.max() + 1)})")

fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [1, 1.6]})
ax = axes[0]
ax.fill_between(clim.index, clim.p10, clim.p90, color="#f5b041", alpha=0.35, label="10–90 % of years")
ax.plot(clim.index, clim["median"], color="#ca6f1e", lw=2, label="Median year")
last = weekly.index.max()
ax.plot(weekly.columns, weekly.loc[last], color="#c0392b", lw=1.5, label=str(last))
ax.axvspan(critical_weeks.min() - 0.5, critical_weeks.max() + 0.5, color="red", alpha=0.07, label="Critical period")
ax.set(yscale="symlog", xlim=(1, 52), xticks=(month_starts.dayofyear - 1) / 7 + 1, ylabel="fire cells / week",
       title=f"Weekly climatology: {AOI_NAME}")
ax.set_xticklabels(month_starts.strftime("%b")); ax.legend(loc="upper left")

ax = axes[1]
cmap_z = plt.get_cmap("RdBu_r").copy(); cmap_z.set_bad("#dddddd")
im = ax.imshow(z.values, aspect="auto", cmap=cmap_z, vmin=-4, vmax=4,
               extent=[0.5, 52.5, z.index.max() + 0.5, z.index.min() - 0.5])
ax.set_xticks((month_starts.dayofyear - 1) / 7 + 1, month_starts.strftime("%b")); ax.set_yticks(z.index)
ax.set(title="Anomaly calendar (robust z-score vs. same week in other years)", ylabel="Year")
plt.colorbar(im, ax=ax, pad=0.01, label="z  (red = more fire than usual)")
plt.tight_layout(); plt.show()

In [ ]:
# Season timing and annual summary per year
def season_stats(g):
    c = g.cumsum(); tot = c.iloc[-1]
    if tot == 0:
        return pd.Series({"total": 0, "start_10%": pd.NaT, "peak_50%": pd.NaT, "end_90%": pd.NaT, "peak_day": pd.NaT})
    f = lambda q: c.index[np.searchsorted(c.values, q * tot)]
    return pd.Series({"total": tot, "start_10%": f(.1), "peak_50%": f(.5), "end_90%": f(.9), "peak_day": g.idxmax()})

season = aoi_daily.groupby(aoi_daily.index.year).apply(season_stats).unstack()
for col in ["start_10%", "peak_50%", "end_90%", "peak_day"]:
    season[col] = pd.to_datetime(season[col]).dt.strftime("%d %b")
season["total"] = season["total"].astype(float).round(0)
season["vs median"] = (season["total"] / season["total"].median()).round(2)
season["source"] = np.where(season.index < viirs_start.year, "MODIS (cal.)", TARGET)
display(season)

top = z.stack().dropna().sort_values(ascending=False).head(10)
print("Most unusual weeks (highest z):")
for (y, wk_), v in top.items():
    print(f"  {y} week {wk_:>2} (from {week_to_date(wk_)}): z = {v:+.1f}, {weekly.loc[y, wk_]:,.0f} fire cells")

## 8. Red-pixel fire map
Every harmonized 0.1° fire cell for `MAP_YEAR`, drawn as a red pixel. Brighter means more fire days in that cell. On the left, every year stacked shows where fire burns over and over.

In [ ]:
def red_map(df, ax, title):
    # Rasterize harmonized fire cells onto the common grid: one red pixel per cell
    la0, la1, lo0, lo1 = AOI
    la1 = min(la1, 72)                                   # almost no fire north of 72°N
    y0, y1 = int(np.floor(la0 / GRID_DEG)), int(np.ceil(la1 / GRID_DEG))
    x0, x1 = int(np.floor(lo0 / GRID_DEG)), int(np.ceil(lo1 / GRID_DEG))
    g = df.groupby(["cy", "cx"]).weight.sum().reset_index()
    g = g[(g.cy >= y0) & (g.cy < y1) & (g.cx >= x0) & (g.cx < x1)]
    grid = np.full((y1 - y0, x1 - x0), np.nan, dtype="float32")
    grid[g.cy - y0, g.cx - x0] = np.log1p(g.weight)
    cmap = plt.get_cmap("Reds").copy(); cmap.set_bad("#111418")
    ax.imshow(grid, origin="lower", extent=[x0 * GRID_DEG, x1 * GRID_DEG, y0 * GRID_DEG, y1 * GRID_DEG],
              cmap=cmap, vmin=0, vmax=np.nanpercentile(grid, 99) if g.size else 1, interpolation="nearest")
    ax.set(title=title, xlabel="Longitude", ylabel="Latitude")
    ax.set_aspect(1 / np.cos(np.deg2rad((la0 + la1) / 2)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), facecolor="white")
red_map(aoi_cells, axes[0], f"All years {harm.year.min()}–{harm.year.max()} ({AOI_NAME})")
red_map(aoi_cells[aoi_cells.year == MAP_YEAR], axes[1], f"{MAP_YEAR} ({AOI_NAME})")
plt.tight_layout(); plt.show()

## 9. Export for the web app
* `harmonized_cells.csv.gz`: one row per 0.1° fire cell-day (date, lat, lon, weight, FRP, source). The web app filters this by AOI.
* `daily_<AOI>.csv`: harmonized daily series for the current AOI
* `calendar_weekly_<AOI>.csv` / `anomaly_z_<AOI>.csv`: year × week matrices behind the calendar heatmaps
* `calibration_factors.csv`: monthly k-factors, so the method can be reproduced

In [ ]:
slug = AOI_NAME.lower().replace(" ", "_")
export = harm[["acq_date", "lat", "lon", "weight", "n_det", "frp_sum", "source"]].copy()
export["lat"] = export["lat"].round(2); export["lon"] = export["lon"].round(2)
export.to_csv(OUT_DIR / "harmonized_cells.csv.gz", index=False, compression="gzip")
aoi_daily.rename("fire_cells").to_csv(OUT_DIR / f"daily_{slug}.csv")
weekly.to_csv(OUT_DIR / f"calendar_weekly_{slug}.csv")
z.round(2).to_csv(OUT_DIR / f"anomaly_z_{slug}.csv")
k_month.to_csv(OUT_DIR / "calibration_factors.csv")
season.to_csv(OUT_DIR / f"season_summary_{slug}.csv")
for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:35s} {p.stat().st_size/1e6:8.2f} MB")

In [ ]:
# Map pins: one row per 0.1° cell per year, with a summary for the pin's popup
g = harm.sort_values("acq_date").groupby(["year", "lat", "lon"], observed=True)
pins = g.agg(fire_days=("weight", "size"),            # days with fire in this cell
             harmonized_cells=("weight", "sum"),      # fire_days on the TARGET scale (MODIS × k before 2012)
             detections=("n_det", "sum"),             # raw satellite detections
             frp_total_mw=("frp_sum", "sum"),         # summed fire radiative power (MW)
             frp_max_day_mw=("frp_sum", "max"),       # most intense day
             first_fire=("acq_date", "first"),
             last_fire=("acq_date", "last"),
             source=("source", "first")).reset_index()
pins["peak_day"] = harm.loc[g["frp_sum"].idxmax().values, "acq_date"].values
for c in ["first_fire", "last_fire", "peak_day"]:
    pins[c] = pd.to_datetime(pins[c]).dt.strftime("%Y-%m-%d")
pins["lat"] = pins["lat"].round(2); pins["lon"] = pins["lon"].round(2)
pins[["harmonized_cells", "frp_total_mw", "frp_max_day_mw"]] = pins[["harmonized_cells", "frp_total_mw", "frp_max_day_mw"]].round(1)
pins.to_csv(OUT_DIR / "map_pins_yearly.csv", index=False)
print(f"map_pins_yearly.csv: {len(pins):,} pins")
pins.head()

## Notes and limitations
* **Fire cell-days, not burned area.** One cell-day means a fire was burning somewhere in that ~10 km cell that day. It is not a measure of how much land burned.
* **Calibration assumes the MODIS/VIIRS relationship is stable through time.** The hold-out test (4c) tells you how much to trust that for 2002–2011.
* **Only MODIS Aqua is in the table.** Adding Terra (10:30 overpass) would roughly double MODIS sampling but would need its own k-factors.
* **VIIRS low-confidence detections** were already absent from the source files, so `MIN_CONF = "l"` only changes MODIS.
* **The AOI presets are rough bounding boxes**, not exact borders. For exact province or ecozone boundaries, use a shapefile (e.g. with geopandas).